# 17. POS Tagging Evaluation

## Overview
A POS tagger should be evaluated against manually annotated reference tags. This notebook builds a simple statistical tagger, evaluates it on held-out tagged sentences, and examines accuracy, coverage, and common errors.

The Brown corpus is used because it provides human-annotated English text. The evaluation uses the universal tagset to keep tag labels compact.

## 1. Load the tagged corpus

Download the Brown corpus and inspect the available data.

In [1]:
import nltk

nltk.download("brown")
nltk.download("universal_tagset")

from nltk.corpus import brown

tagged_sentences = brown.tagged_sents(tagset="universal")
print("Number of tagged sentences:", len(tagged_sentences))
print("Example:", tagged_sentences[0][:12])

[nltk_data] Downloading package brown to /home/mg/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     /home/mg/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


Number of tagged sentences: 57340
Example: [('The', 'DET'), ('Fulton', 'NOUN'), ('County', 'NOUN'), ('Grand', 'ADJ'), ('Jury', 'NOUN'), ('said', 'VERB'), ('Friday', 'NOUN'), ('an', 'DET'), ('investigation', 'NOUN'), ('of', 'ADP'), ("Atlanta's", 'NOUN'), ('recent', 'ADJ')]


## 2. Prepare train and test data

The split is deterministic so that the example can be reproduced. In a production experiment, use a carefully designed split that matches the target domain.

In [2]:
from sklearn.model_selection import train_test_split

train_sents, test_sents = train_test_split(
    tagged_sentences,
    test_size=0.2,
    random_state=42,
)

print(f"Training sentences: {len(train_sents):,}")
print(f"Test sentences:     {len(test_sents):,}")

Training sentences: 45,872
Test sentences:     11,468


## 3. Build taggers with backoff

A unigram tagger predicts the most common tag observed for each word. A bigram tagger can use the previous predicted tag; when it cannot make a prediction, it backs off to the unigram tagger, then to a default tagger.

In [3]:
from nltk.tag import DefaultTagger, UnigramTagger, BigramTagger

default_tagger = DefaultTagger("NOUN")
unigram_tagger = UnigramTagger(train_sents, backoff=default_tagger)
bigram_tagger = BigramTagger(train_sents, backoff=unigram_tagger)

print("Taggers are ready.")

Taggers are ready.


## 4. Evaluate token accuracy

Token accuracy is the proportion of tokens whose predicted tag matches the reference tag. Evaluate each model on the same held-out sentences.

In [4]:
def token_accuracy(tagger, gold_sentences):
    correct = 0
    total = 0

    for sentence in gold_sentences:
        words = [word for word, _ in sentence]
        gold_tags = [tag for _, tag in sentence]
        predicted = tagger.tag(words)
        predicted_tags = [tag for _, tag in predicted]

        correct += sum(g == p for g, p in zip(gold_tags, predicted_tags))
        total += len(gold_tags)

    return correct / total if total else 0.0

results = {
    "Default tagger": token_accuracy(default_tagger, test_sents),
    "Unigram tagger": token_accuracy(unigram_tagger, test_sents),
    "Bigram + backoff": token_accuracy(bigram_tagger, test_sents),
}

for model_name, score in results.items():
    print(f"{model_name:<20} accuracy = {score:.4f}")

Default tagger       accuracy = 0.2364
Unigram tagger       accuracy = 0.9452
Bigram + backoff     accuracy = 0.9532


## 5. Evaluate unknown-word coverage

Coverage is the percentage of test tokens for which the model predicts a tag without falling back to the default tag. Here, the default tag is `NOUN`, so this is an approximate diagnostic rather than a universal definition of coverage.

In [5]:
def non_default_prediction_rate(tagger, gold_sentences, default_tag="NOUN"):
    predicted_total = 0
    non_default_total = 0

    for sentence in gold_sentences:
        words = [word for word, _ in sentence]
        predicted_tags = [tag for _, tag in tagger.tag(words)]
        predicted_total += len(predicted_tags)
        non_default_total += sum(tag != default_tag for tag in predicted_tags)

    return non_default_total / predicted_total if predicted_total else 0.0

for name, tagger in [
    ("Default tagger", default_tagger),
    ("Unigram tagger", unigram_tagger),
    ("Bigram + backoff", bigram_tagger),
]:
    rate = non_default_prediction_rate(tagger, test_sents)
    print(f"{name:<20} non-default prediction rate = {rate:.4f}")

Default tagger       non-default prediction rate = 0.0000
Unigram tagger       non-default prediction rate = 0.7528
Bigram + backoff     non-default prediction rate = 0.7532


## 6. Inspect errors with a confusion matrix

A confusion matrix shows which reference tags are being confused with which predicted tags. To keep this demonstration readable, it uses a subset of the test set.

In [6]:
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

gold_flat = []
pred_flat = []

for sentence in test_sents[:500]:
    words = [word for word, _ in sentence]
    gold_tags = [tag for _, tag in sentence]
    pred_tags = [tag for _, tag in bigram_tagger.tag(words)]

    gold_flat.extend(gold_tags)
    pred_flat.extend(pred_tags)

labels = sorted(set(gold_flat) | set(pred_flat))
matrix = confusion_matrix(gold_flat, pred_flat, labels=labels)
confusion = pd.DataFrame(matrix, index=labels, columns=labels)

display(confusion)

,.,ADJ,ADP,ADV,CONJ,DET,NOUN,NUM,PRON,PRT,VERB,X
.,1233,0,0,0,0,0,0,0,0,0,0,0
ADJ,0,643,2,16,0,0,62,0,0,0,6,0
ADP,0,2,1073,10,0,6,2,0,14,102,1,0
ADV,0,12,32,430,2,2,3,0,0,5,1,0
CONJ,0,0,1,0,321,3,0,0,0,0,0,0
DET,0,0,4,0,1,1167,0,0,7,0,0,0
NOUN,0,19,0,1,0,0,2237,1,0,0,43,0
NUM,0,0,0,0,0,0,6,113,0,0,0,0
PRON,0,0,1,0,0,1,1,0,415,0,0,0
PRT,0,2,28,2,0,0,3,0,0,213,0,0


## 7. Per-tag precision, recall, and F1

Overall accuracy can hide weak performance on less frequent tags. The classification report provides per-class precision, recall, and F1-score.

In [7]:
print(classification_report(gold_flat, pred_flat, labels=labels, zero_division=0))

              precision    recall  f1-score   support

           .       1.00      1.00      1.00      1233
         ADJ       0.95      0.88      0.91       729
         ADP       0.94      0.89      0.91      1210
         ADV       0.93      0.88      0.91       487
        CONJ       0.99      0.99      0.99       325
         DET       0.99      0.99      0.99      1179
        NOUN       0.94      0.97      0.95      2301
         NUM       0.99      0.95      0.97       119
        PRON       0.95      0.99      0.97       418
         PRT       0.67      0.86      0.75       248
        VERB       0.97      0.96      0.96      1554
           X       1.00      0.13      0.24        15

    accuracy                           0.95      9818
   macro avg       0.94      0.87      0.88      9818
weighted avg       0.95      0.95      0.95      9818



## Interpretation and limitations

- Evaluation must use data that was not used to fit the tagger.
- Token accuracy is intuitive but can be dominated by frequent tags.
- Per-tag precision, recall, and F1 reveal performance on individual categories.
- Random sentence splitting can place similar material in both sets; domain-aware evaluation may be more realistic.
- These results measure a simple corpus-trained baseline, not the performance of every pretrained POS tagger.

## Summary

This notebook demonstrates a reproducible evaluation workflow: load gold annotations, split the data, train baseline taggers, measure held-out accuracy, and inspect class-level errors.